# Figure Regeneration — Phase 2 (feature-selection configs)

Rebuilds the full per-config diagnostic figure suite for the nine feature-selection / regime
configs, matching the Phase-1 architecture notebook's plots, using only the saved per-patient
predictions (`per_patient_sealed_predictions_FS.json`: `y_true` + `probs`). No GPU, no rerun.

Per config: confusion matrix, ROC, PR, DET, lift, precision/recall-vs-threshold, calibration.
Across configs: ROC overlay, PR overlay, metric-comparison bars, MCC ranking.

Excluded: t-SNE — it needs the node embeddings, which were not saved. It cannot be
regenerated from predictions alone; note this rather than approximate it.

In [ ]:
# CELL 1 — IMPORTS
import json, os, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score,
                             matthews_corrcoef, f1_score, confusion_matrix)
from sklearn.calibration import calibration_curve
plt.rcParams.update({'figure.dpi':150,'savefig.bbox':'tight','font.size':10})

In [ ]:
# CELL 2 — CONFIG + LOAD
# On Kaggle set FS_PATH to your dataset path; locally leave the filename.
FS_PATH = 'per_patient_sealed_predictions_FS.json'
OUT     = 'phase2_figures/'
os.makedirs(OUT, exist_ok=True)

fs = json.load(open(FS_PATH))
CONFIGS = list(fs.keys())
y = np.asarray(fs['BASELINE']['y_true'])
P = {k: np.asarray(v['probs']) for k, v in fs.items()}
assert all(len(v)==len(y) for v in P.values())
print(f"Loaded {len(CONFIGS)} configs | n={len(y)} | positives={int(y.sum())}")

In [ ]:
# CELL 3 — HELPERS
def hard(p): return (p>=0.5).astype(int)
def det_points(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return fpr, 1-tpr  # FPR vs FNR
def lift_points(y, p, bins=10):
    order=np.argsort(-p); yy=y[order]; n=len(yy); base=yy.mean()
    xs=[]; ys=[]
    for i in range(1,bins+1):
        cut=int(np.ceil(n*i/bins)); xs.append(i/bins*100)
        ys.append((yy[:cut].mean()/base) if base>0 else 0)
    return xs, ys
def metrics_of(y, p):
    h=hard(p); fpr,tpr,_=roc_curve(y,p)
    tn,fp,fn,tp=confusion_matrix(y,h).ravel()
    return dict(MCC=matthews_corrcoef(y,h), AUROC=auc(fpr,tpr),
                AUPRC=average_precision_score(y,p), F1=f1_score(y,h),
                Accuracy=(tp+tn)/(tp+tn+fp+fn),
                Sensitivity=tp/(tp+fn), Specificity=tn/(tn+fp))

In [ ]:
# CELL 4 — PER-CONFIG DIAGNOSTIC PLOTS
for cfg in CONFIGS:
    p=P[cfg]; h=hard(p)
    # confusion
    cm=confusion_matrix(y,h)
    fig,ax=plt.subplots(figsize=(4,3.4))
    im=ax.imshow(cm,cmap='Blues')
    for (i,j),v in np.ndenumerate(cm): ax.text(j,i,int(v),ha='center',va='center',
        color='white' if v>cm.max()/2 else 'black',fontsize=12)
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['Pred 0','Pred 1'])
    ax.set_yticklabels(['True 0','True 1']); ax.set_title(f'{cfg} — Confusion')
    fig.savefig(f'{OUT}{cfg}_confusion.png'); plt.close(fig)
    # ROC + PR
    fpr,tpr,_=roc_curve(y,p); pr,rc,_=precision_recall_curve(y,p)
    fig,axes=plt.subplots(1,2,figsize=(8,3.4))
    axes[0].plot(fpr,tpr); axes[0].plot([0,1],[0,1],'--',color='grey')
    axes[0].set_title(f'{cfg} ROC (AUC={auc(fpr,tpr):.3f})'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[1].plot(rc,pr); axes[1].set_title(f'{cfg} PR (AP={average_precision_score(y,p):.3f})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    fig.savefig(f'{OUT}{cfg}_roc_pr.png'); plt.close(fig)
    # DET
    dfpr,dfnr=det_points(y,p)
    fig,ax=plt.subplots(figsize=(4,3.4)); ax.plot(dfpr,dfnr)
    ax.set_xlabel('FPR'); ax.set_ylabel('FNR'); ax.set_title(f'{cfg} — DET'); fig.savefig(f'{OUT}{cfg}_det.png'); plt.close(fig)
    # lift
    lx,ly=lift_points(y,p)
    fig,ax=plt.subplots(figsize=(4,3.4)); ax.plot(lx,ly,marker='o'); ax.axhline(1,ls='--',color='grey')
    ax.set_xlabel('% targeted'); ax.set_ylabel('Lift'); ax.set_title(f'{cfg} — Lift'); fig.savefig(f'{OUT}{cfg}_lift.png'); plt.close(fig)
    # precision/recall vs threshold
    ths=np.linspace(0.05,0.95,19)
    precs=[]; recs=[]
    for t in ths:
        ht=(p>=t).astype(int); tp=((ht==1)&(y==1)).sum()
        precs.append(tp/max((ht==1).sum(),1)); recs.append(tp/max((y==1).sum(),1))
    fig,ax=plt.subplots(figsize=(4,3.4)); ax.plot(ths,precs,label='Precision'); ax.plot(ths,recs,label='Recall')
    ax.axvline(0.5,ls='--',color='grey'); ax.set_xlabel('Threshold'); ax.legend(); ax.set_title(f'{cfg} — P/R vs threshold')
    fig.savefig(f'{OUT}{cfg}_prec_thresh.png'); plt.close(fig)
    # calibration
    try:
        frac,mean=calibration_curve(y,p,n_bins=8,strategy='quantile')
        fig,ax=plt.subplots(figsize=(4,3.4)); ax.plot(mean,frac,marker='o'); ax.plot([0,1],[0,1],'--',color='grey')
        ax.set_xlabel('Mean predicted'); ax.set_ylabel('Observed'); ax.set_title(f'{cfg} — Calibration')
        fig.savefig(f'{OUT}{cfg}_calibration.png'); plt.close(fig)
    except Exception as e:
        print(f'calibration skipped for {cfg}: {e}')
print('per-config plots done')

In [ ]:
# CELL 5 — CROSS-CONFIG OVERLAYS (ROC, PR)
fig,ax=plt.subplots(figsize=(6,5))
for cfg in CONFIGS:
    fpr,tpr,_=roc_curve(y,P[cfg]); ax.plot(fpr,tpr,label=f'{cfg} ({auc(fpr,tpr):.3f})',lw=1)
ax.plot([0,1],[0,1],'--',color='grey'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC overlay — all configs'); ax.legend(fontsize=7); fig.savefig(f'{OUT}ALL_roc_overlay.png'); plt.close(fig)

fig,ax=plt.subplots(figsize=(6,5))
for cfg in CONFIGS:
    pr,rc,_=precision_recall_curve(y,P[cfg]); ax.plot(rc,pr,label=f'{cfg} ({average_precision_score(y,P[cfg]):.3f})',lw=1)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR overlay — all configs')
ax.legend(fontsize=7); fig.savefig(f'{OUT}ALL_pr_overlay.png'); plt.close(fig)
print('overlays done')

In [ ]:
# CELL 6 — METRIC COMPARISON BARS + MCC RANKING
M={cfg:metrics_of(y,P[cfg]) for cfg in CONFIGS}
metrics=['MCC','AUROC','AUPRC','F1','Accuracy','Sensitivity','Specificity']
x=np.arange(len(CONFIGS)); w=0.11
fig,ax=plt.subplots(figsize=(12,5))
for i,mt in enumerate(metrics):
    ax.bar(x+i*w,[M[c][mt] for c in CONFIGS],w,label=mt)
ax.set_xticks(x+w*3); ax.set_xticklabels(CONFIGS,rotation=45,ha='right')
ax.legend(fontsize=8,ncol=7,loc='lower center',bbox_to_anchor=(0.5,1.02)); ax.set_ylim(0,1)
ax.set_title('Metric comparison — all configs'); fig.savefig(f'{OUT}ALL_metric_comparison.png'); plt.close(fig)

order=sorted(CONFIGS,key=lambda c:M[c]['MCC'],reverse=True)
fig,ax=plt.subplots(figsize=(7,5)); ax.barh([o for o in order][::-1],[M[o]['MCC'] for o in order][::-1])
ax.set_xlabel('MCC'); ax.set_title('MCC ranking'); fig.savefig(f'{OUT}ALL_mcc_ranking.png'); plt.close(fig)
print('ranking done | best:',order[0],round(M[order[0]]['MCC'],4))

In [ ]:
# CELL 7 — ZIP ALL FIGURES
import zipfile
zp=f'{OUT.rstrip("/")}_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        if f.endswith('.png'): z.write(os.path.join(OUT,f),f)
n=len([f for f in os.listdir(OUT) if f.endswith('.png')])
print(f'zipped {n} figures -> {zp}')

## Notes
- Every figure derives from `y_true` + `probs`; identical inputs to the statistical-tests notebook, so numbers are consistent across both.
- **t-SNE excluded**: requires node embeddings, not saved. To add it later, save the final-layer embeddings per config in a future run, then t-SNE can be drawn without re-training.
- On Kaggle, set `FS_PATH` to the uploaded JSON's dataset path; everything else is path-independent.